In [10]:
import re
import pandas as pd
import json
import os

In [23]:
import os
import re
import json
import pandas as pd

def read_ann_files(directory):
    ann_files = [f for f in os.listdir(directory) if f.endswith('.ann')]
    data = {}
    for file in ann_files:
        file_path = os.path.join(directory, file)
        with open(file_path, 'r', encoding='utf-8') as f:
            data[file] = f.read()
    return data

def extract_text_and_offsets(details):
    parts = details.split('\t')
    entity_info = parts[0].split(' ')
    text_content = parts[-1]
    offsets = ' '.join([part for part in entity_info[1:] if part.isdigit()])
    return text_content, offsets

def parse_ann_file(content):
    lines = content.strip().split('\n')
    df = pd.DataFrame([line.split('\t', 1) for line in lines], columns=['ID', 'Details'])
    df['Text'], df['Offsets'] = zip(*df['Details'].apply(extract_text_and_offsets))
    t_e_df = df[df['ID'].str.startswith(('T', 'E'))]
    text_dict = t_e_df.set_index('ID')['Text'].to_dict()
    offset_dict = t_e_df.set_index('ID')['Offsets'].to_dict()
    rel_df = df[df['ID'].str.startswith('R')]
    #rel_pattern = re.compile(r'^R\d+\t.*?(And|Or).*$')
    rel_pattern = re.compile(r'^(R\d+)\t(And|Or) Arg1:(E\d+|T\d+) Arg2:(E\d+|T\d+)$')
    relationships = []
    for index, row in rel_df.iterrows():
        line = f"{row['ID']}\t{row['Details']}"
        rel_match = rel_pattern.match(line)
        print(rel_match)
        if rel_match:
            rel_id, rel_details = line.split('\t', 1)
            arg1_match = re.search(r'Arg1:(E\d+|T\d+)', rel_details)
            arg2_match = re.search(r'Arg2:(E\d+|T\d+)', rel_details)
            if arg1_match and arg2_match:
                rel_type = rel_match.group(1)
                arg1 = arg1_match.group(1)
                arg2 = arg2_match.group(1)
                relationships.append((rel_type, arg1, arg2))

    neg_df = df[df['Details'].str.contains('Negation')]
    negations = neg_df[['ID', 'Text', 'Offsets']].to_dict(orient='records')
    return text_dict, offset_dict, relationships, negations

def get_full_text(text_dict, offset_dict, entity_id):
    if entity_id in text_dict:
        text_content = text_dict[entity_id]
        offsets = offset_dict[entity_id]
        if entity_id.startswith('E'):
            sub_entity_id = re.search(r'\b(T\d+)\b', text_content)
            if sub_entity_id:
                sub_text, sub_offsets = get_full_text(text_dict, offset_dict, sub_entity_id.group(1))
                return sub_text, sub_offsets
        return text_content, offsets
    return entity_id, ""

def create_rel_texts(text_dict, offset_dict, relationships, negations):
    rel_texts = []
    for rel_type, arg1, arg2 in relationships:
        arg1_text, arg1_offset = get_full_text(text_dict, offset_dict, arg1)
        arg2_text, arg2_offset = get_full_text(text_dict, offset_dict, arg2)
        rel_texts.append({
            "type": rel_type,
            "arg1_text": arg1_text,
            "arg1_offset": arg1_offset,
            "arg2_text": arg2_text,
            "arg2_offset": arg2_offset
        })
    for neg in negations:
        if 'Negation' in neg['Text']:
            continue
        rel_texts.append({
            "type": "Negation",
            "arg1_text": neg['Text'],
            "arg1_offset": neg['Offsets'],
            "arg2_text": "",
            "arg2_offset": ""
        })
    return rel_texts

def save_results(rel_texts, file_prefix, output_directory):
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    file_name = f"{file_prefix}.json"
    file_path = os.path.join(output_directory, file_name)
    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(rel_texts, f, indent=4)

ann_directory = 'lct_ann'
output_directory = 'all_operators2'

ann_data = read_ann_files(ann_directory)
for file, content in ann_data.items():
    if file == "NCT03925467.ann":
        text_dict, offset_dict, relationships, negations = parse_ann_file(content)
        rel_texts = create_rel_texts(text_dict, offset_dict, relationships, negations)
        save_results(rel_texts, file.replace('.ann', ''), output_directory)

None
<re.Match object; span=(0, 22), match='R2\tOr Arg1:E11 Arg2:E9'>
<re.Match object; span=(0, 23), match='R3\tOr Arg1:E32 Arg2:E31'>
<re.Match object; span=(0, 23), match='R4\tOr Arg1:E35 Arg2:E34'>
<re.Match object; span=(0, 23), match='R5\tOr Arg1:E39 Arg2:E40'>
<re.Match object; span=(0, 23), match='R6\tOr Arg1:E25 Arg2:E23'>
<re.Match object; span=(0, 23), match='R7\tOr Arg1:E42 Arg2:E25'>
<re.Match object; span=(0, 23), match='R8\tOr Arg1:E43 Arg2:E42'>
None


In [21]:
ann_directory = 'lct_ann'
output_directory = 'all_operators2'

### Find exact different operators found with different regex

In [25]:
def parse_ann_file(content):
    lines = content.strip().split('\n')
    df = pd.DataFrame([line.split('\t', 1) for line in lines], columns=['ID', 'Details'])
    df['Text'], df['Offsets'] = zip(*df['Details'].apply(extract_text_and_offsets))
    t_e_df = df[df['ID'].str.startswith(('T', 'E'))]
    text_dict = t_e_df.set_index('ID')['Text'].to_dict()
    offset_dict = t_e_df.set_index('ID')['Offsets'].to_dict()
    rel_df = df[df['ID'].str.startswith('R')]

    rel_pattern1 = re.compile(r'^R\d+\t.*?(And|Or).*$')
    rel_pattern2 = re.compile(r'^(R\d+)\t(And|Or) Arg1:(E\d+|T\d+) Arg2:(E\d+|T\d+)$')

    found_diff = False

    for index, row in rel_df.iterrows():
        line = f"{row['ID']}\t{row['Details']}"
        if rel_pattern1.match(line):
            rel_match1 = rel_pattern1.match(line)
            rel_type1 = rel_match1.group(1)
            rel_match2 = rel_pattern2.match(line)
            if not rel_match2 or rel_match2.group(2) != rel_type1:
                print(rel_match1)
                found_diff = True
                break

    neg_df = df[df['Details'].str.contains('Negation')]
    negations = neg_df[['ID', 'Text', 'Offsets']].to_dict(orient='records')

    if found_diff:
        return text_dict, offset_dict, [], negations, True
    else:
        return text_dict, offset_dict, [], negations, False

ann_directory = 'lct_ann'
output_directory = 'all_operators2'

ann_data = read_ann_files(ann_directory)
filenames = []

for file, content in ann_data.items():
    text_dict, offset_dict, relationships, negations, found_diff = parse_ann_file(content)
    if found_diff:
        filenames.append(file)

print("Dateinamen mit Operatoren, die nur mit rel_pattern1 gefunden wurden:")
for filename in filenames:
    print(filename)

<re.Match object; span=(0, 24), match='R13\tOr Arg1:E18 Arg2:E7\t'>
<re.Match object; span=(0, 24), match='R1\tOr Arg1:E15 Arg2:E13\t'>
<re.Match object; span=(0, 23), match='R8\tOr Arg1:E27 Arg2:E3\t'>
<re.Match object; span=(0, 25), match='R14\tOr Arg1:E12 Arg2:E40\t'>
<re.Match object; span=(0, 25), match='R9\tAnd Arg1:E29 Arg2:T65\t'>
<re.Match object; span=(0, 25), match='R14\tAnd Arg1:E2 Arg2:T27\t'>
<re.Match object; span=(0, 24), match='R15\tAnd Arg1:E3 Arg2:E1\t'>
<re.Match object; span=(0, 22), match='R7\tOr Arg1:E3 Arg2:E9\t'>
Dateinamen mit Operatoren, die nur mit rel_pattern1 gefunden wurden:
NCT03863353.ann
NCT03867981.ann
NCT03920488.ann
NCT03924934.ann
NCT03925467.ann
NCT03929640.ann
NCT03931941.ann
NCT03934346.ann
